In [1]:
# Funciones auxiliares

import numpy as np

def house(x):
  '''
  u, rho = house(x)
  Calcula u y rho tal que Q = I - rho u u^T
  cumple Qx = |x|_2 e^1
  '''
  n = len(x)
  rho = 0
  u = x.copy()
  u[0] = 1.

  if n == 1:
    sigma = 0
  else:
    sigma = np.sum(x[1:]**2)

  if sigma>0 or x[0]<0:
    mu = np.sqrt(x[0]**2 + sigma)
    if x[0]<=0:
      gamma = x[0] - mu
    else:
      gamma = -sigma/(x[0] + mu)

    rho = 2*gamma**2/(gamma**2 + sigma)
    u = u/gamma
    u[0] = 1

  return u, rho

def givens(x1,x2):
    '''
    c, s = givens(x1, x2)
    Calcula el coseno y seno para la rotación de Givens
    que hace (x1,x2) -> (y,0).
    '''
    c = 1.
    s = 0.
    ax1 = abs(x1)
    ax2 = abs(x2)
    if ax1 + ax2 > 0:
        if ax2 > ax1:
            tau = -x1/x2
            s = -np.sign(x2)/np.sqrt(1 + tau**2)
            c = tau*s
        else:
            tau = -x2/x1
            c = np.sign(x1)/np.sqrt(1 + tau**2)
            s = tau*c
    return c, s

#Ejercicio 13

In [5]:
def fhess(A, p):
    m, n = A.shape
    if m != n:
        print("La matriz no es cuadrada")
        return None
    Q = np.eye(m)
    H = A.copy()

    if p == 0: #Realiza ref Householder
        for j in range(n-2):
            #I = j+1: , J =j:
            u, rho = house(H[j+1:, j])
            w = rho * u
            H[j+1:, j:] = H[j+1:, j:] - np.outer(w, u.T @ H[j+1:, j:])
            H[:, j+1:] = H[:, j+1:] - H[:, j+1:]@ np.outer(w, u.T)
            Q[:, j+1:] = Q[:, j+1:] - Q[:, j+1:]@np.outer(w, u.T)

    elif p == 1: # Realiza rot Givens
        for j in range(n - 2):
            for i in range(j + 2, n):
                c, s = givens(H[j + 1, j], H[i, j])
                rot = np.array([[c, -s], [s, c]])
                H[[j + 1, i], j:] = rot @ H[[j + 1, i], j:]
                H[:, [j + 1, i]] = H[:, [j + 1, i]] @ rot.T
                Q[:, [j + 1, i]] = Q[:, [j + 1, i]] @ rot.T
    else:
        print("Elegir un p que sea 0 o 1")
        return None

    return Q, H

# TEST de House

A = np.random.random((5,5))
Q_h, H_h = fhess(A, 0)

print("TEST Householder")
print(f"Matriz ortogonal Q = {Q_h}")
print("---------------------------------------------------------------")
print(f"Hessenberg superior H ={H_h}")
print("---------------------------------------------------------------")
print(f"||A-Q @ H @ Q.T||_fro = {np.linalg.norm(A-Q_h @ H_h @ Q_h.T)}")
print("---------------------------------------------------------------")
print()
###############################################################################

# TEST de Givens
A = np.random.random((5,5))
Q_g, H_g = fhess(A, 1)

print()
print("TEST Givens")
print(f"Matriz ortogonal Q = {Q_g}")
print("-------------------------------------------------------------")
print(f"Hessenberg superior H ={H_g}")
print("-------------------------------------------------------------")
print(f"||A-Q @ H @ Q.T||_fro = {np.linalg.norm(A-Q_g @ H_g @ Q_g.T)}")
print("-------------------------------------------------------------")

TEST Householder
Matriz ortogonal Q = [[ 1.          0.          0.          0.          0.        ]
 [ 0.          0.19931547  0.86858218  0.22749308  0.39253693]
 [ 0.          0.65926289 -0.45421692  0.16042482  0.57734159]
 [ 0.          0.60637586  0.19742068 -0.68827459 -0.34584892]
 [ 0.          0.39743439  0.01664516  0.66991706 -0.62688115]]
---------------------------------------------------------------
Hessenberg superior H =[[ 4.04127652e-01  1.25700111e+00 -3.37443864e-02  4.64992676e-01
  -5.38608170e-04]
 [ 1.05332286e+00  1.45351277e+00 -7.37274133e-02  5.51346026e-01
   1.14976906e-01]
 [-1.11022302e-16  9.06455537e-01  5.72618957e-01  4.82154035e-01
  -4.17570104e-01]
 [ 0.00000000e+00 -2.22044605e-16  4.08457983e-01 -1.28313617e-01
   8.83196080e-02]
 [-5.55111512e-17 -1.11022302e-16 -5.55111512e-17  1.30876067e-01
   8.47628846e-02]]
---------------------------------------------------------------
||A-Q @ H @ Q.T||_fro = 9.573913561248061e-16
-----------------------